# 01 · Chunking, Embedding and the Index

The corpus is small and synthetic on purpose: every fact lives in exactly one
document, so the evaluation labels are genuinely correct rather than estimated.
That makes recall@k measurable instead of arguable.

This notebook does the two stages that decide everything — chunking and
indexing — and shows what a bad chunking choice costs before you notice it.

In [1]:
import json, re, textwrap
import numpy as np, pandas as pd
from pathlib import Path

pd.set_option("display.width", 130); pd.set_option("display.max_colwidth", 70)

ROOT = Path.cwd()
while not (ROOT / "data" / "corpus").exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
CORPUS = ROOT / "data" / "corpus"
DOCS = json.loads((CORPUS / "manifest.json").read_text())
EVAL = json.loads((ROOT / "data" / "corpus_eval.json").read_text())
print(f"{len(DOCS)} documents, {len(EVAL)} eval questions")

10 documents, 10 eval questions


In [2]:
for d in DOCS[:3]:
    print(f"--- {d['doc_id']}  ({d['kind']})  {len(d['text'].split())} words")
print("\nkinds:", pd.Series([d["kind"] for d in DOCS]).value_counts().to_dict())

--- site-ard  (site)  72 words
--- site-whl  (site)  70 words
--- site-cly  (site)  70 words

kinds: {'site': 4, 'incident': 2, 'standard': 1, 'policy': 1, 'report': 1, 'procedure': 1}


## Two chunking strategies

The naive one splits on a fixed character count. The structural one splits on
markdown headings, which is what a self-contained thought looks like in this
corpus.

Both are defensible in the abstract. Read the output and only one of them is.

In [3]:
def chunk_fixed(text, size=350, overlap=50):
    out, i = [], 0
    while i < len(text):
        out.append(text[i:i+size])
        i += size - overlap
    return out

def chunk_structural(text):
    parts = re.split(r"\n(?=#{1,3} )", text)
    out = []
    for p in parts:
        p = p.strip()
        if not p:
            continue
        if len(p) > 900:                      # long section: split on blank lines
            out += [s.strip() for s in re.split(r"\n\n+", p) if s.strip()]
        else:
            out.append(p)
    return out

sample = next(d for d in DOCS if d["doc_id"] == "ops-std-04")["text"]
print("FIXED-SIZE — first three chunks\n")
for c in chunk_fixed(sample)[:3]:
    print(repr(c[:150]), "\n")

FIXED-SIZE — first three chunks

'# OPS-STD-04 — Standard Operating Envelope\n\n## Gearbox temperature\n\nThe gearbox oil temperature must not exceed 82 degrees Celsius during\ncontinuous o' 

'## Vibration\n\nNacelle vibration above 0.55 mm/s RMS sustained for more than thirty\nminutes triggers a Level 2 alert. Above 0.80 mm/s the turbine shuts' 

'pm.' 



In [4]:
print("STRUCTURAL — first three chunks\n")
for c in chunk_structural(sample)[:3]:
    print(repr(c[:150]), "\n")

STRUCTURAL — first three chunks

'# OPS-STD-04 — Standard Operating Envelope' 

'## Gearbox temperature\n\nThe gearbox oil temperature must not exceed 82 degrees Celsius during\ncontinuous operation. Between 82 and 90 degrees the turb' 

'## Vibration\n\nNacelle vibration above 0.55 mm/s RMS sustained for more than thirty\nminutes triggers a Level 2 alert. Above 0.80 mm/s the turbine shuts' 



### Read those two outputs

The fixed-size chunks start and end mid-sentence. One of them begins with
`degrees the turbine enters` — that fragment has lost the number it refers to
and the condition it applies under. Embed it and you get a vector for a
half-thought.

The structural chunks are whole sections. Each one can be read alone.

**This is the ten minutes the lesson said not to skip.**

In [5]:
chunks = []
for d in DOCS:
    for i, c in enumerate(chunk_structural(d["text"])):
        chunks.append({"chunk_id": f"{d['doc_id']}#{i}", "doc_id": d["doc_id"],
                       "kind": d["kind"], "text": c})
chunks = pd.DataFrame(chunks)
print(f"{len(chunks)} chunks from {len(DOCS)} documents")
print(f"words per chunk: median {chunks['text'].str.split().str.len().median():.0f}, "
      f"max {chunks['text'].str.split().str.len().max()}")
chunks.head(3)[["chunk_id","kind"]]

36 chunks from 10 documents
words per chunk: median 22, max 43


,chunk_id,kind
0,site-ard#0,site
1,site-ard#1,site
2,site-ard#2,site


## Embedding

A small local model — no API key, and it runs offline. 384 dimensions.

In [6]:
from sentence_transformers import SentenceTransformer

encoder = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")
vecs = encoder.encode(chunks["text"].tolist(), normalize_embeddings=True,
                      show_progress_bar=False)
print("embedding matrix:", vecs.shape)

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

embedding matrix: (36, 384)


### Sanity-check the embeddings before trusting them

If a paraphrase does not score higher than an unrelated sentence, something is
wrong with the model or the text, and no amount of retrieval tuning will help.

In [7]:
probe = encoder.encode([
    "maximum gearbox oil temperature",       # 0
    "how hot can the gearbox get",           # 1 — paraphrase of 0
    "who manages the Clyde site",            # 2 — unrelated
], normalize_embeddings=True)

print(f"paraphrase pair   : {float(probe[0] @ probe[1]):.3f}")
print(f"unrelated pair    : {float(probe[0] @ probe[2]):.3f}")
print("✅ sensible" if probe[0] @ probe[1] > probe[0] @ probe[2] + 0.15
      else "❌ embeddings are not separating meaning — stop here")

paraphrase pair   : 0.725
unrelated pair    : -0.064
✅ sensible


## Into pgvector

A real index rather than an in-memory list, because the lesson about index
recall only means something against a real index.

Requires the container: `make up` or `docker compose -f docker/compose.yaml up -d pgvector`.

In [8]:
import psycopg2
from psycopg2.extras import execute_values

DSN = "host=localhost port=5433 dbname=aiml user=aiml password=aiml"
try:
    conn = psycopg2.connect(DSN); conn.autocommit = True
    HAVE_PG = True
    print("connected to pgvector")
except Exception as e:
    HAVE_PG = False
    print("pgvector not reachable — the notebook falls back to numpy exact search")
    print(f"  ({e.__class__.__name__})  start it with: docker compose -f docker/compose.yaml up -d pgvector")

connected to pgvector


In [9]:
if HAVE_PG:
    with conn.cursor() as cur:
        cur.execute("CREATE EXTENSION IF NOT EXISTS vector")
        cur.execute("DROP TABLE IF EXISTS chunks")
        cur.execute("""CREATE TABLE chunks (
                         chunk_id text PRIMARY KEY,
                         doc_id   text,
                         kind     text,
                         body     text,
                         embedding vector(384))""")
        execute_values(cur,
            "INSERT INTO chunks (chunk_id, doc_id, kind, body, embedding) VALUES %s",
            [(r.chunk_id, r.doc_id, r.kind, r.text, list(map(float, v)))
             for r, v in zip(chunks.itertuples(), vecs)])
        cur.execute("SELECT count(*) FROM chunks")
        print(f"{cur.fetchone()[0]} chunks indexed")

36 chunks indexed


### An index, and the parameter that decides recall

`ef_search` is the dial to move when recall is low — raise it before rebuilding
with a larger `M`.

In [10]:
if HAVE_PG:
    with conn.cursor() as cur:
        cur.execute("""CREATE INDEX chunks_hnsw ON chunks
                       USING hnsw (embedding vector_cosine_ops)
                       WITH (m = 16, ef_construction = 64)""")
        cur.execute("SET hnsw.ef_search = 40")
        cur.execute("""SELECT chunk_id, 1 - (embedding <=> %s::vector) AS score
                       FROM chunks ORDER BY embedding <=> %s::vector LIMIT 3""",
                    (list(map(float, encoder.encode("gearbox temperature limit",
                                                    normalize_embeddings=True))),) * 2)
        for cid, sc in cur.fetchall():
            print(f"  {sc:.3f}  {cid}")

  0.748  ops-std-04#1
  0.478  site-ard#1
  0.476  site-bat#1


## Save the artefacts for the next notebook

In [11]:
np.save(ROOT / "data" / "chunk_vecs.npy", vecs)
chunks.to_csv(ROOT / "data" / "chunks.csv", index=False)
print(f"saved {len(chunks)} chunks and their embeddings")

saved 36 chunks and their embeddings


---

## Your turn

1. Re-run with `chunk_fixed` instead of `chunk_structural` and keep the file.
   Notebook 02 will measure what it costs you.
2. Change `ef_search` to 4 and re-run the query. Do the top results change?
3. Print ten random chunks and read them. Would you be able to answer a question
   from any one of them alone?